# 03 — Double Exponential Smoothing (Holt's Method)

**Reference:** Vandeput, N. (2021). *Data Science for Supply Chain Forecasting* (2nd ed.), Chapter 5 & 7.

Double exponential smoothing (Holt's method) adds a **trend component** to SES.
Two equations — one for the level, one for the trend:

$$l_t = \alpha \cdot d_t + (1 - \alpha)(l_{t-1} + b_{t-1})$$
$$b_t = \beta(l_t - l_{t-1}) + (1 - \beta) b_{t-1}$$
$$f_{t+m} = l_t + m \cdot b_t$$

Where:
- `l` = level (smoothed demand)
- `b` = trend (smoothed slope)
- `α` = level smoothing parameter
- `β` = trend smoothing parameter
- `m` = periods ahead


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

## 1. The Model

In [ ]:
def double_exp_smoothing(d, extra_periods=12, alpha=0.4, beta=0.4):
    """
    Double exponential smoothing — Holt's linear method.
    Captures level + trend. No seasonality.

    Parameters
    ----------
    d : array-like
    extra_periods : int
    alpha : float — level smoothing
    beta : float — trend smoothing
    """
    cols = len(d)
    d = np.append(d, [np.nan] * extra_periods)
    f = np.full(cols + extra_periods, np.nan)
    l_arr = np.full(cols + extra_periods, np.nan)
    b_arr = np.full(cols + extra_periods, np.nan)

    # Initialise
    l_arr[0] = d[0]
    b_arr[0] = d[1] - d[0]
    f[0] = d[0]

    for t in range(1, cols):
        f[t] = l_arr[t - 1] + b_arr[t - 1]
        l_arr[t] = alpha * d[t] + (1 - alpha) * f[t]
        b_arr[t] = beta * (l_arr[t] - l_arr[t - 1]) + (1 - beta) * b_arr[t - 1]

    # Future forecast: extrapolate trend
    for m in range(1, extra_periods + 1):
        f[cols + m - 1] = l_arr[cols - 1] + m * b_arr[cols - 1]

    return pd.DataFrame({
        'Demand': d, 'Forecast': f, 'Level': l_arr,
        'Trend': b_arr, 'Error': d - f
    })

## 2. Simulated Trending Demand

In [ ]:
np.random.seed(42)
t = np.arange(48)
# Upward trend of 15 units/month with noise
demand = 800 + 15 * t + np.random.normal(0, 60, 48)
demand = np.clip(demand, 0, None)

df = double_exp_smoothing(demand, extra_periods=12, alpha=0.3, beta=0.2)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df['Demand'], label='Actual Demand', color='#2C2C2A', linewidth=1.5)
ax.plot(df['Forecast'], label='Double ES Forecast', color='#185FA5',
        linestyle='--', linewidth=1.5)
ax.axvline(x=len(demand), color='gray', linestyle=':', alpha=0.5)
ax.set_title('Double Exponential Smoothing — Trend Capture', fontsize=13)
ax.set_xlabel('Period (months)')
ax.set_ylabel('Demand (units)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Damped Trend (Chapter 7)

A pure trend will extrapolate forever — often unrealistic.
The **damped trend** adds a damping factor `φ` that shrinks the trend over time.

$$f_{t+m} = l_t + (\phi + \phi^2 + ... + \phi^m) \cdot b_t$$


In [ ]:
def damped_double_exp_smoothing(d, extra_periods=12, alpha=0.4, beta=0.3, phi=0.9):
    """
    Damped trend exponential smoothing.
    phi < 1 dampens the trend — forecast flattens over time.
    phi = 1 reduces to standard Holt's method.
    """
    cols = len(d)
    d = np.append(d, [np.nan] * extra_periods)
    f = np.full(cols + extra_periods, np.nan)
    l_arr = np.full(cols + extra_periods, np.nan)
    b_arr = np.full(cols + extra_periods, np.nan)

    l_arr[0] = d[0]
    b_arr[0] = d[1] - d[0]
    f[0] = d[0]

    for t in range(1, cols):
        f[t] = l_arr[t - 1] + phi * b_arr[t - 1]
        l_arr[t] = alpha * d[t] + (1 - alpha) * f[t]
        b_arr[t] = beta * (l_arr[t] - l_arr[t - 1]) + (1 - beta) * phi * b_arr[t - 1]

    for m in range(1, extra_periods + 1):
        damping = sum(phi ** i for i in range(1, m + 1))
        f[cols + m - 1] = l_arr[cols - 1] + damping * b_arr[cols - 1]

    return pd.DataFrame({'Demand': d, 'Forecast': f, 'Error': d - f})


df_damped = damped_double_exp_smoothing(demand, extra_periods=24, phi=0.85)
df_nodamp = double_exp_smoothing(demand, extra_periods=24, alpha=0.3, beta=0.2)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df_nodamp['Demand'], label='Actual Demand', color='#2C2C2A', linewidth=1.5)
ax.plot(df_nodamp['Forecast'], label='Holt (no damping)', color='#D85A30',
        linestyle='--')
ax.plot(df_damped['Forecast'], label='Damped trend (φ=0.85)', color='#185FA5',
        linestyle='--')
ax.axvline(x=len(demand), color='gray', linestyle=':', alpha=0.5,
           label='Forecast horizon')
ax.set_title('Damped vs Undamped Trend — Long-horizon Forecasting', fontsize=13)
ax.set_xlabel('Period (months)')
ax.set_ylabel('Demand (units)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Key Takeaways

| Model | Captures | Use When |
|-------|----------|----------|
| SES | Level only | Stable demand, no trend |
| Holt's (DES) | Level + trend | Clear upward/downward trend |
| Damped Holt's | Level + fading trend | Trend exists but expected to slow |

**Next:** Triple exponential smoothing (`04_triple_exponential_smoothing.ipynb`) adds seasonality.
